In [1]:
# ============================================================
# MD12 — ROBERTA-BASE + ALPHA 0.75 + 4 EPOCHS
#
# Research question:
#   MD11 RoBERTa-base reached its best Macro-F1 at epoch 3 and
#   was still improving from epoch 2 to epoch 3. Does a longer
#   4-epoch schedule allow RoBERTa-base to overtake MD6?
#
# ONLY MAJOR CHANGE FROM MD11:
#   epochs: 3 -> 4
#
# Everything else is held fixed:
#   Model: FacebookAI/roberta-base
#   Max length: 512
#   Class weighting: inverse-frequency weights ** 0.75
#   Learning rate: 2e-5
#   Effective global batch: 16
#   Per-device batch: 4
#   GPUs: 2
#   Gradient accumulation: 2
#   Weight decay: 0.01
#   Warmup steps: 280
#   Seed: 42
#
# IMPORTANT:
#   This is a fresh 4-epoch run. With the default linear learning-
#   rate scheduler, increasing total epochs also stretches the LR
#   decay schedule. We do not resume MD11 because its 3-epoch LR
#   schedule has already decayed to its endpoint.
#
# Primary metric: Macro-F1
# Secondary metric: Accuracy
# Test set: NOT USED
# ============================================================

import gc
import json
import math
import random
import shutil
from pathlib import Path
from collections import Counter

import numpy as np
import torch
import torch.nn.functional as F

from datasets import Dataset
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

SEED = 42
MODEL_NAME = "FacebookAI/roberta-base"
MAX_LENGTH = 512
WEIGHT_ALPHA = 0.75
LEARNING_RATE = 2e-5

PER_DEVICE_TRAIN_BATCH = 4
GRAD_ACCUM = 2
EXPECTED_GPUS = 2
EFFECTIVE_BATCH = (
    PER_DEVICE_TRAIN_BATCH * EXPECTED_GPUS * GRAD_ACCUM
)
EPOCHS = 4

TRAIN_PATH = Path(
    "/kaggle/input/datasets/malindus/mathdial-move-selection/train.jsonl"
)
VAL_PATH = Path(
    "/kaggle/input/datasets/malindus/mathdial-move-selection/validation.jsonl"
)

OUTPUT_DIR = Path("/kaggle/working/md12_roberta_base_weighted_alpha075_4epochs")
RESULT_DIR = Path("/kaggle/working/md12_results")
RESULT_DIR.mkdir(parents=True, exist_ok=True)

LABELS = ["generic", "probing", "focus", "telling"]
LABEL2ID = {label: i for i, label in enumerate(LABELS)}
ID2LABEL = {i: label for label, i in LABEL2ID.items()}

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("=" * 78)
print("MD12 CONFIGURATION")
print("=" * 78)
print("Model:", MODEL_NAME)
print("Max length:", MAX_LENGTH)
print("Weight alpha:", WEIGHT_ALPHA)
print("Learning rate:", LEARNING_RATE)
print("Per-device batch:", PER_DEVICE_TRAIN_BATCH)
print("Gradient accumulation:", GRAD_ACCUM)
print("Expected GPUs:", EXPECTED_GPUS)
print("Effective global batch:", EFFECTIVE_BATCH)
print("Epochs:", EPOCHS)
print("Detected GPU count:", torch.cuda.device_count())

assert torch.cuda.is_available(), "GPU is required."
assert torch.cuda.device_count() == EXPECTED_GPUS, (
    "MD12 is defined for Kaggle 2xT4. "
    f"Detected {torch.cuda.device_count()} GPU(s)."
)
assert EFFECTIVE_BATCH == 16

def load_jsonl(path):
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

train_rows = load_jsonl(TRAIN_PATH)
val_rows = load_jsonl(VAL_PATH)

assert len(train_rows) == 14905
assert len(val_rows) == 1850

required = {"example_id", "qid", "problem", "history", "target_move"}

for name, rows in [
    ("train", train_rows),
    ("validation", val_rows),
]:
    assert required.issubset(rows[0].keys()), f"{name} schema mismatch"
    bad = [r for r in rows if r["target_move"] not in LABEL2ID]
    assert not bad, f"{name}: invalid labels found"

print("Train examples:", len(train_rows))
print("Validation examples:", len(val_rows))
print("Test set used: False")


MD12 CONFIGURATION
Model: FacebookAI/roberta-base
Max length: 512
Weight alpha: 0.75
Learning rate: 2e-05
Per-device batch: 4
Gradient accumulation: 2
Expected GPUs: 2
Effective global batch: 16
Epochs: 4
Detected GPU count: 2
Train examples: 14905
Validation examples: 1850
Test set used: False


In [2]:
# ============================================================
# INPUT FORMAT + TOKENIZATION + FIXED LENGTH DIAGNOSTIC
# ============================================================

def format_history(history):
    if not history:
        return "[No previous conversation]"

    return "\n".join(
        f"{turn['user']}: {turn['text']}"
        for turn in history
    )

def build_problem_text(example):
    return "Problem:\n" + str(example["problem"])

def build_conversation_text(example):
    return (
        "Conversation:\n"
        + format_history(example["history"])
        + "\n\nNext teacher pedagogical move:"
    )

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.truncation_side = "left"

def prepare_rows(rows):
    prepared = []

    for ex in rows:
        prepared.append({
            "example_id": ex["example_id"],
            "qid": ex["qid"],
            "problem_text": build_problem_text(ex),
            "conversation_text": build_conversation_text(ex),
            "labels": LABEL2ID[ex["target_move"]],
        })

    return prepared

train_prepared = prepare_rows(train_rows)
val_prepared = prepare_rows(val_rows)

# Use the original DistilRoBERTa-defined length split so MD12
# diagnostics are directly comparable with MD6/MD8/MD9/MD10.
distil_tokenizer = AutoTokenizer.from_pretrained(
    "distilbert/distilroberta-base"
)
distil_tokenizer.truncation_side = "left"

val_original_lengths = []

for ex in val_rows:
    encoded = distil_tokenizer(
        build_problem_text(ex),
        build_conversation_text(ex),
        add_special_tokens=True,
        truncation=False,
    )
    val_original_lengths.append(len(encoded["input_ids"]))

val_original_lengths = np.asarray(val_original_lengths)

short_mask = val_original_lengths <= 512
long_mask = val_original_lengths > 512

print("Validation <=512:", int(short_mask.sum()))
print("Validation >512:", int(long_mask.sum()))

assert int(short_mask.sum()) == 1507
assert int(long_mask.sum()) == 343

train_ds = Dataset.from_list(train_prepared)
val_ds = Dataset.from_list(val_prepared)

def tokenize_batch(batch):
    return tokenizer(
        batch["problem_text"],
        batch["conversation_text"],
        truncation="only_second",
        max_length=MAX_LENGTH,
    )

train_tok = train_ds.map(
    tokenize_batch,
    batched=True,
    remove_columns=[
        "problem_text",
        "conversation_text",
        "example_id",
        "qid",
    ],
)

val_tok = val_ds.map(
    tokenize_batch,
    batched=True,
    remove_columns=[
        "problem_text",
        "conversation_text",
        "example_id",
        "qid",
    ],
)

assert max(len(x) for x in train_tok["input_ids"]) <= MAX_LENGTH
assert max(len(x) for x in val_tok["input_ids"]) <= MAX_LENGTH

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print("Tokenization: PASSED")


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Token indices sequence length is longer than the specified maximum sequence length for this model (578 > 512). Running this sequence through the model will result in indexing errors


Validation <=512: 1507
Validation >512: 343


Map:   0%|          | 0/14905 [00:00<?, ? examples/s]

Map:   0%|          | 0/1850 [00:00<?, ? examples/s]

Tokenization: PASSED


In [3]:
# ============================================================
# ALPHA=0.75 CLASS WEIGHTS
# ============================================================

train_counts = Counter(r["target_move"] for r in train_rows)

n = len(train_rows)
k = len(LABELS)

raw_class_weights = np.asarray(
    [
        n / (k * train_counts[label])
        for label in LABELS
    ],
    dtype=np.float32,
)

class_weights = np.power(
    raw_class_weights,
    WEIGHT_ALPHA,
).astype(np.float32)

print("=" * 78)
print("MD12 CLASS WEIGHTS")
print("=" * 78)

for label, raw_w, tuned_w in zip(
    LABELS,
    raw_class_weights,
    class_weights,
):
    print(
        f"{label:10s} count={train_counts[label]:4d} "
        f"raw={raw_w:.6f} alpha075={tuned_w:.6f}"
    )


MD12 CLASS WEIGHTS
generic    count=3579 raw=1.041143 alpha075=1.030701
probing    count=3413 raw=1.091781 alpha075=1.068075
focus      count=5443 raw=0.684595 alpha075=0.752619
telling    count=2470 raw=1.508603 alpha075=1.361229


In [4]:
# ============================================================
# MODEL + WEIGHTED TRAINER + TRAINING
# ============================================================

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABELS),
    label2id=LABEL2ID,
    id2label=ID2LABEL,
)

class WeightedTrainer(Trainer):
    def __init__(
        self,
        *args,
        class_weights=None,
        **kwargs,
    ):
        super().__init__(*args, **kwargs)

        self.class_weights_tensor = torch.tensor(
            class_weights,
            dtype=torch.float32,
        )

    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        num_items_in_batch=None,
    ):
        labels = inputs.pop("labels")

        outputs = model(**inputs)
        logits = outputs.logits

        weights = self.class_weights_tensor.to(
            logits.device
        )

        loss = F.cross_entropy(
            logits,
            labels,
            weight=weights,
        )

        if return_outputs:
            return loss, outputs

        return loss

def compute_metrics(eval_prediction):
    logits, labels = eval_prediction

    pred = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy_score(labels, pred),
        "macro_f1": f1_score(
            labels,
            pred,
            labels=list(range(len(LABELS))),
            average="macro",
            zero_division=0,
        ),
    }

steps_per_epoch = math.ceil(
    len(train_rows) / EFFECTIVE_BATCH
)

print("Expected optimizer steps/epoch:", steps_per_epoch)
print("Expected total optimizer steps:", steps_per_epoch * EPOCHS)

assert steps_per_epoch == 932

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),

    learning_rate=LEARNING_RATE,

    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=GRAD_ACCUM,

    num_train_epochs=EPOCHS,

    weight_decay=0.01,
    warmup_steps=280,

    eval_strategy="epoch",
    logging_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=4,

    # Evaluate every epoch checkpoint manually at the end.
    load_best_model_at_end=False,

    fp16=True,

    seed=SEED,
    data_seed=SEED,

    report_to="none",
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,

    train_dataset=train_tok,
    eval_dataset=val_tok,

    processing_class=tokenizer,
    data_collator=data_collator,

    compute_metrics=compute_metrics,
    class_weights=class_weights,
)

print("Trainer GPU count:", trainer.args.n_gpu)

assert trainer.args.n_gpu == EXPECTED_GPUS

print("\n" + "=" * 78)
print("STARTING MD12 TRAINING")
print("=" * 78)

trainer.train()

history_path = RESULT_DIR / "md12_log_history.json"

with history_path.open("w", encoding="utf-8") as f:
    json.dump(
        trainer.state.log_history,
        f,
        indent=2,
    )

print("\n" + "=" * 78)
print("EPOCH VALIDATION HISTORY")
print("=" * 78)

for row in trainer.state.log_history:
    if "eval_macro_f1" in row:
        print(
            f"Epoch {row.get('epoch')}: "
            f"accuracy={row.get('eval_accuracy'):.6f} "
            f"macro_f1={row.get('eval_macro_f1'):.6f} "
            f"val_loss={row.get('eval_loss'):.6f}"
        )

print("\nAll 4 epoch checkpoints saved.")
print(
    "Best checkpoint will be selected manually "
    "by validation Macro-F1."
)


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: FacebookAI/roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Expected optimizer steps/epoch: 932
Expected total optimizer steps: 3728
Trainer GPU count: 2

STARTING MD12 TRAINING


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,2.469533,1.218720,0.435135,0.437073
2,2.313737,1.181517,0.488108,0.478425
3,2.163871,1.193574,0.500541,0.481745
4,1.990214,1.225797,0.508108,0.479561


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


EPOCH VALIDATION HISTORY
Epoch 1.0: accuracy=0.435135 macro_f1=0.437073 val_loss=1.218720
Epoch 2.0: accuracy=0.488108 macro_f1=0.478425 val_loss=1.181517
Epoch 3.0: accuracy=0.500541 macro_f1=0.481745 val_loss=1.193574
Epoch 4.0: accuracy=0.508108 macro_f1=0.479561 val_loss=1.225797

All 4 epoch checkpoints saved.
Best checkpoint will be selected manually by validation Macro-F1.


In [5]:
# ==========================================================
# EVALUATE ALL CHECKPOINTS + SAVE BEST MODEL
# ==========================================================

def softmax_np(logits):
    z = logits - logits.max(
        axis=1,
        keepdims=True,
    )

    ez = np.exp(z)

    return ez / ez.sum(
        axis=1,
        keepdims=True,
    )

def subset_metrics(
    true_ids,
    pred_ids,
    mask,
):
    y_true = true_ids[mask]
    y_pred = pred_ids[mask]

    return {
        "n": int(mask.sum()),
        "accuracy": float(
            accuracy_score(y_true, y_pred)
        ),
        "macro_f1": float(
            f1_score(
                y_true,
                y_pred,
                labels=list(range(len(LABELS))),
                average="macro",
                zero_division=0,
            )
        ),
    }

checkpoint_dirs = sorted(
    OUTPUT_DIR.glob("checkpoint-*"),
    key=lambda p: int(
        p.name.split("-")[-1]
    ),
)

assert len(checkpoint_dirs) == 4, (
    f"Expected 4 checkpoints, found {len(checkpoint_dirs)}"
)

all_results = []

for epoch_idx, ckpt in enumerate(
    checkpoint_dirs,
    start=1,
):
    print("\n" + "-" * 78)
    print(f"EPOCH {epoch_idx}: {ckpt}")
    print("-" * 78)

    ckpt_model = (
        AutoModelForSequenceClassification
        .from_pretrained(ckpt)
    )

    eval_args = TrainingArguments(
        output_dir=str(
            RESULT_DIR / f"eval_epoch{epoch_idx}"
        ),
        per_device_eval_batch_size=8,
        fp16=True,
        seed=SEED,
        data_seed=SEED,
        report_to="none",
    )

    eval_trainer = WeightedTrainer(
        model=ckpt_model,
        args=eval_args,

        eval_dataset=val_tok,

        processing_class=tokenizer,
        data_collator=data_collator,

        compute_metrics=compute_metrics,
        class_weights=class_weights,
    )

    pred_output = eval_trainer.predict(
        val_tok
    )

    logits = pred_output.predictions
    true_ids = pred_output.label_ids

    pred_ids = np.argmax(
        logits,
        axis=1,
    )

    probs = softmax_np(logits)

    overall_accuracy = float(
        accuracy_score(
            true_ids,
            pred_ids,
        )
    )

    overall_macro_f1 = float(
        f1_score(
            true_ids,
            pred_ids,
            labels=list(range(len(LABELS))),
            average="macro",
            zero_division=0,
        )
    )

    per_class_f1 = f1_score(
        true_ids,
        pred_ids,
        labels=list(range(len(LABELS))),
        average=None,
        zero_division=0,
    )

    report = classification_report(
        true_ids,
        pred_ids,
        labels=list(range(len(LABELS))),
        target_names=LABELS,
        digits=6,
        zero_division=0,
        output_dict=True,
    )

    cm = confusion_matrix(
        true_ids,
        pred_ids,
        labels=list(range(len(LABELS))),
    ).tolist()

    short_result = subset_metrics(
        true_ids,
        pred_ids,
        short_mask,
    )

    long_result = subset_metrics(
        true_ids,
        pred_ids,
        long_mask,
    )

    result = {
        "epoch": epoch_idx,
        "checkpoint": str(ckpt),

        "accuracy": overall_accuracy,
        "macro_f1": overall_macro_f1,

        "per_class_f1": {
            label: float(score)
            for label, score
            in zip(
                LABELS,
                per_class_f1,
            )
        },

        "classification_report": report,
        "confusion_matrix": cm,

        "le512": short_result,
        "gt512": long_result,
    }

    all_results.append(result)

    print("Accuracy:", overall_accuracy)
    print("Macro-F1:", overall_macro_f1)
    print(
        "<=512 Macro-F1:",
        short_result["macro_f1"],
    )
    print(
        ">512 Macro-F1:",
        long_result["macro_f1"],
    )
    print(
        "Per-class F1:",
        result["per_class_f1"],
    )

    pred_path = (
        RESULT_DIR
        / f"md12_epoch{epoch_idx}_predictions.jsonl"
    )

    with pred_path.open(
        "w",
        encoding="utf-8",
    ) as f:
        for i, (
            true_id,
            pred_id,
            probability_row,
        ) in enumerate(
            zip(
                true_ids,
                pred_ids,
                probs,
            )
        ):
            record = {
                "example_id": (
                    val_rows[i]["example_id"]
                ),
                "qid": val_rows[i]["qid"],

                "true_move": (
                    ID2LABEL[int(true_id)]
                ),
                "predicted_move": (
                    ID2LABEL[int(pred_id)]
                ),

                "original_distil_token_length": int(
                    val_original_lengths[i]
                ),

                "probabilities": {
                    label: float(
                        probability_row[j]
                    )
                    for j, label
                    in enumerate(LABELS)
                },
            }

            f.write(
                json.dumps(record)
                + "\n"
            )

    del eval_trainer
    del ckpt_model

    gc.collect()
    torch.cuda.empty_cache()

best_result = max(
    all_results,
    key=lambda x: x["macro_f1"],
)

summary = {
    "experiment": (
        "MD12_RoBERTaBase_weighted_CE_alpha075_4epochs"
    ),

    "research_question": (
        "Can extending the MD11 RoBERTa-base configuration from "
        "3 to 4 epochs improve validation Macro-F1 and beat MD6?"
    ),

    "model": MODEL_NAME,
    "max_length": MAX_LENGTH,
    "weight_alpha": WEIGHT_ALPHA,
    "learning_rate": LEARNING_RATE,
    "effective_global_batch": EFFECTIVE_BATCH,
    "epochs": EPOCHS,
    "test_used": False,

    "class_weights": {
        label: float(w)
        for label, w
        in zip(
            LABELS,
            class_weights,
        )
    },

    "baseline_md6_epoch2": {
        "accuracy": 0.5016216216216216,
        "macro_f1": 0.487684572655523,
    },

    "md10_modernbert_epoch2": {
        "accuracy": 0.49513513513513513,
        "macro_f1": 0.4694130591189515,
    },

    "baseline_md11_epoch3": {
        "accuracy": 0.5021621621621621,
        "macro_f1": 0.48428708496304046,
    },

    "historical_hybrid_reference": {
        "accuracy": 0.5005405405,
        "macro_f1": 0.4867677775,
    },

    "epochs_results": all_results,

    "best_epoch": best_result["epoch"],
    "best_accuracy": best_result["accuracy"],
    "best_macro_f1": best_result["macro_f1"],
}

summary_path = (
    RESULT_DIR
    / "md12_summary.json"
)

with summary_path.open(
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        summary,
        f,
        indent=2,
    )

best_checkpoint = Path(
    best_result["checkpoint"]
)

best_model = (
    AutoModelForSequenceClassification
    .from_pretrained(best_checkpoint)
)

best_model_dir = (
    RESULT_DIR
    / "md12_best_model"
)

best_model.save_pretrained(
    best_model_dir
)

tokenizer.save_pretrained(
    best_model_dir
)

archive_path = shutil.make_archive(
    str(
        RESULT_DIR
        / "md12_best_model"
    ),
    "gztar",
    root_dir=best_model_dir,
)

print("\n" + "=" * 78)
print("MD12 COMPLETE")
print("=" * 78)

print(
    "Best epoch:",
    best_result["epoch"],
)

print(
    "Best accuracy:",
    best_result["accuracy"],
)

print(
    "Best Macro-F1:",
    best_result["macro_f1"],
)

print(
    "Summary:",
    summary_path,
)

print(
    "Best model archive:",
    archive_path,
)

print("\nDecision references:")
print(
    "MD6 Epoch 2 accuracy = "
    "0.5016216216216216"
)
print(
    "MD6 Epoch 2 Macro-F1 = "
    "0.487684572655523"
)
print(
    "MD10 Epoch 2 Macro-F1 = "
    "0.4694130591189515"
)
print(
    "MD11 Epoch 3 accuracy = "
    "0.5021621621621621"
)
print(
    "MD11 Epoch 3 Macro-F1 = "
    "0.48428708496304046"
)



------------------------------------------------------------------------------
EPOCH 1: /kaggle/working/md12_roberta_base_weighted_alpha075_4epochs/checkpoint-932
------------------------------------------------------------------------------


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Accuracy: 0.43513513513513513
Macro-F1: 0.4370732344568262
<=512 Macro-F1: 0.45435479731664097
>512 Macro-F1: 0.14117829778422034
Per-class F1: {'generic': 0.5597826086956522, 'probing': 0.3413333333333333, 'focus': 0.44873949579831934, 'telling': 0.3984375}

------------------------------------------------------------------------------
EPOCH 2: /kaggle/working/md12_roberta_base_weighted_alpha075_4epochs/checkpoint-1864
------------------------------------------------------------------------------


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Accuracy: 0.4881081081081081
Macro-F1: 0.478425023702574
<=512 Macro-F1: 0.4778803037517634
>512 Macro-F1: 0.24503894420199984
Per-class F1: {'generic': 0.5985037406483791, 'probing': 0.42824074074074076, 'focus': 0.5305199698568199, 'telling': 0.3564356435643564}

------------------------------------------------------------------------------
EPOCH 3: /kaggle/working/md12_roberta_base_weighted_alpha075_4epochs/checkpoint-2796
------------------------------------------------------------------------------


...

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Accuracy: 0.5081081081081081
Macro-F1: 0.4795611167374287
<=512 Macro-F1: 0.47757371885612665
>512 Macro-F1: 0.3570731234952013
Per-class F1: {'generic': 0.6106696935300795, 'probing': 0.41379310344827586, 'focus': 0.5800561797752809, 'telling': 0.3137254901960784}


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


MD12 COMPLETE
Best epoch: 3
Best accuracy: 0.5005405405405405
Best Macro-F1: 0.4817452965264059
Summary: /kaggle/working/md12_results/md12_summary.json
Best model archive: /kaggle/working/md12_results/md12_best_model.tar.gz

Decision references:
MD6 Epoch 2 accuracy = 0.5016216216216216
MD6 Epoch 2 Macro-F1 = 0.487684572655523
MD10 Epoch 2 Macro-F1 = 0.4694130591189515
MD11 Epoch 3 accuracy = 0.5021621621621621
MD11 Epoch 3 Macro-F1 = 0.48428708496304046
